In [7]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from skimage.io import imread
from skimage.util import img_as_float
%matplotlib inline

1. Загрузите картинку parrots.jpg. Преобразуйте изображение, приведя все значения в интервал от 0 до 1. Для этого можно воспользоваться функцией img_as_float из модуля skimage. Обратите внимание на этот шаг, так как при работе с исходным изображением вы получите некорректный результат.

In [8]:
image = imread('parrots.jpg')
image_float = img_as_float(image)


2. Создайте матрицу объекты-признаки: характеризуйте каждый пиксель тремя координатами - значениями интенсивности в пространстве RGB

In [9]:
h, w, c = image_float.shape
pixels = image_float.reshape(-1, c)  

3. Запустите алгоритм K-Means с параметрами init=’k-means++’ и random_state=241. После выделения кластеров все пиксели, отнесенные в один кластер, попробуйте заполнить двумя способами: медианным и средним цветом по кластеру.

In [10]:
def compress_image(pixels, kmeans, method='mean'):
    labels = kmeans.labels_
    compressed = np.zeros_like(pixels)
    for cluster_id in range(kmeans.n_clusters):
        mask = labels == cluster_id
        if method == 'mean':
            color = pixels[mask].mean(axis=0)
        else:
            color = np.median(pixels[mask], axis=0)
        compressed[mask] = color
    return compressed


results = []

for n_clusters in range(1, 21):
    km = KMeans(n_clusters=n_clusters, init='k-means++', random_state=241, n_init=10)
    km.fit(pixels)
    results.append({
        'n_clusters': n_clusters,
        'compressed_mean':   compress_image(pixels, km, method='mean'),
        'compressed_median': compress_image(pixels, km, method='median'),
    })

4. Измерьте качество получившейся сегментации с помощью метрики PSNR. Эту метрику нужно реализовать самостоятельно (см. определение).

In [11]:
def psnr(original, compressed):
    mse = np.mean((original - compressed) ** 2)
    if mse == 0:
        return float('inf')
    return 10.0 * np.log10(1.0 / mse)

for r in results:
    r['psnr_mean']   = psnr(pixels, r['compressed_mean'])
    r['psnr_median'] = psnr(pixels, r['compressed_median'])

5. Найдите минимальное количество кластеров, при котором значение PSNR выше 20 (можно рассмотреть не более 20 кластеров, но не забудьте рассмотреть оба способа заполнения пикселей одного кластера). Это число и будет ответом в данной задаче.

In [12]:
for r in results:
    if r['psnr_mean'] > 20 or r['psnr_median'] > 20:
        print(r['n_clusters'])
        break

11
